# 03. Entrenar modelos y generar predicciones

Este notebook ejecuta el ajuste supervisado local de T5S, BARTO, mBART-50 y uMT5-Base. Usa solamente los CSV congelados creados por el notebook 02. No emplea APIs de generación ni scripts externos.

**Importante:** `test.csv` no participa en el entrenamiento ni en la selección del checkpoint; solo se usa para generar las predicciones que evaluará el notebook 04.

## Requisitos

Con el entorno virtual activo, instala una vez las dependencias: `pip install -e '.[analysis,models,dev]'`. Los pesos base se descargarán desde Hugging Face la primera vez y se guardarán en su caché local.

**GPU RTX 50 (por ejemplo RTX 5070):** necesitan PyTorch 2.7 o superior compilado con CUDA 12.8. En Windows instala torch con `pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu128` dentro de `.venv`. La celda de validación muestra la versión y la GPU detectada.

## Variantes del experimento

El mismo conjunto de datos y los mismos modelos se entrenan por dos caminos:

| Variante | Qué cambia | Fuente en la carpeta de papers |
| --- | --- | --- |
| `epocas_5` | Ajuste completo, 5 épocas. Solo en `claims_500` reutiliza `claims_500_v1` | #99 (uMT5, 5 épocas) |
| `epocas_10` | Ajuste completo, 10 épocas | #100 (T5, 10 épocas) |
| `epocas_20` | Ajuste completo, 20 épocas | #80 (T5, 20 épocas) |
| `lora` | Solo se entrenan adaptadores LoRA: r=8, alpha=32, dropout=0,1; 10 épocas; learning rate 3e-4 | #75 (configuración), #80 (r=8), #100 (LoRA en T5) |

El learning rate de LoRA coincide con #97. En BARTO, con 3e-5 la pérdida de validación tras 2 épocas fue 2,26 y con 3e-4 fue 1,68. Las demás variantes conservan 3e-5.

Todas usan early stopping con paciencia 3 sobre la pérdida de validación (#66) y conservan el mejor checkpoint. Con *Run All* el notebook entrena las cuatro variantes seguidas, una carpeta de corrida por variante. Para entrenar solo algunas, reduce `VARIANTS_TO_RUN` en la siguiente celda.

**Semilla.** Todas las variantes usan la misma semilla, `BASE_SEED` (20261103), fijada en `docs/EXPERIMENT_PROTOCOL.md`, sección 4.2. Cada variante tiene una sola carpeta de corrida.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

DATASET_NAME = 'claims_1271'  # conjunto cerrado el 26/11/2026 (1017/127/127)
FINAL_DIR = ROOT / 'artifacts' / '02_conjuntos' / DATASET_NAME
from claim_extraction.workflow import BASE_SEED, DATASET_SPLIT_SIZES, EXPERIMENT_VARIANTS, run_name_for

# Con Run All se entrenan las cuatro variantes en este orden: epocas_5, epocas_10, epocas_20 y lora.
# Reduce la lista solo si quieres entrenar algunas, por ejemplo ['lora'].
VARIANTS_TO_RUN = list(EXPERIMENT_VARIANTS)
SEED = BASE_SEED  # semilla única del experimento; no la cambies
EARLY_STOPPING_PATIENCE = 3  # épocas sin mejora en validación antes de detener (paper 66)
RUNS_DIR = ROOT / 'artifacts' / '03_corridas'
# Cada conjunto y variante tiene su propia carpeta de corrida.
RUN_DIRS = {variant: RUNS_DIR / run_name_for(DATASET_NAME, variant) for variant in VARIANTS_TO_RUN}
SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
EXPECTED_SPLIT_SIZES = DATASET_SPLIT_SIZES[DATASET_NAME]
# Parámetros aproximados del checkpoint completo; este flujo ajusta todos sus pesos.
MODEL_IDS = [
    't5s_spanish',  # T5S: ≈248 M parámetros.
    'barto_spanish',  # BARTO: ≈140 M parámetros.
    'mbart50_es',  # mBART-50: ≈610 M parámetros.
    'umt5_base_es',  # uMT5-Base: ≈590 M parámetros.
]
MODELS_TO_RUN = MODEL_IDS  # reduce esta lista para ejecutar solo los modelos pendientes

# Hiperparámetros comunes: sus valores actuales se definen en configs/models.json.
# epochs: los fija cada variante, con early stopping sobre validación.
# learning_rate=3e-5: tamaño de cada actualización de pesos (LoRA usa 3e-4).
# weight_decay=0: penalización de pesos grandes; 0 la desactiva.
# warmup_steps=0: aumento gradual inicial del learning rate; 0 lo desactiva.
# effective_batch_size=8: ejemplos por actualización; con BATCH_SIZE=2 acumula 4 pasos (con 1, 8 pasos).
# max_source_length=512: máximo de tokens de título+cuerpo; el excedente se trunca.
# max_target_length=96: máximo de tokens del claim de referencia durante entrenamiento.
# num_beams=4: alternativas que compara beam search; más beams suele costar más tiempo.
# max_new_tokens=96: límite de tokens al generar el claim.
# Configuración para RTX 5070 (12 GB). Microbatch 2: la GPU procesa 2 ejemplos a la vez y
# acumula 4 pasos, así que cada actualización sigue usando 8 ejemplos (batch efectivo).
# Se eligió 2 para que mBART-50 y uMT5 no se queden sin memoria en una corrida sin supervisión.
BATCH_SIZE = 2
# Precisión mixta bfloat16: más rápida y con menos memoria en GPU RTX 30xx o posteriores.
# Se aplica a todas las variantes y modelos por igual y queda registrada en los manifiestos.
BF16 = True
PROGRESS_EVERY_PERCENT = 5  # usa 1 si quieres ver cada punto porcentual
CACHE_DIR = None  # por ejemplo: Path('/ruta/a/cache_huggingface')
RESUME = True  # True reutiliza checkpoints y predicciones ya completos y reintenta solo lo que falta

# Modelos grandes (≈600 M): recalculan activaciones para caber en 12 GB.
# No cambia los datos, épocas, batch efectivo ni longitudes; solo tarda algo más por la recomputación.
# Con 24 GB o más puede quedar vacío: set().
GRADIENT_CHECKPOINTING_MODELS = {'mbart50_es', 'umt5_base_es'}

print({'variants': VARIANTS_TO_RUN, 'batch_size': BATCH_SIZE, 'bf16': BF16, 'seed': SEED, 'dataset': DATASET_NAME, 'final_dir': str(FINAL_DIR), 'models': MODELS_TO_RUN})

In [ ]:
import json
import platform

from claim_extraction.registry import get_model
from claim_extraction.workflow import (
    available_device,
    bf16_supported,
    generate_predictions,
    release_accelerator_memory,
    train_seq2seq_model,
    utc_now,
    split_fingerprints,
    validate_sem1_splits,
)

def show_training_progress(model_id, percent, current_step, total_steps):
    print(f'{model_id}: {percent}% ({current_step}/{total_steps} pasos)', flush=True)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

if not VARIANTS_TO_RUN or not set(VARIANTS_TO_RUN).issubset(EXPERIMENT_VARIANTS):
    raise ValueError(f'VARIANTS_TO_RUN debe contener variantes de: {", ".join(EXPERIMENT_VARIANTS)}.')
if len(VARIANTS_TO_RUN) != len(set(VARIANTS_TO_RUN)):
    raise ValueError('Cada variante debe aparecer una sola vez.')
if not set(MODELS_TO_RUN).issubset(MODEL_IDS):
    raise ValueError('MODELS_TO_RUN debe contener IDs definidos en MODEL_IDS.')
if len(MODELS_TO_RUN) != len(set(MODELS_TO_RUN)):
    raise ValueError('Cada modelo debe aparecer una sola vez.')
if BATCH_SIZE < 1:
    raise ValueError('BATCH_SIZE debe ser positivo.')
if BF16 and not bf16_supported():
    raise RuntimeError('BF16 = True requiere una GPU CUDA compatible. Revisa la instalación de PyTorch con CUDA o usa BF16 = False.')
if not 1 <= PROGRESS_EVERY_PERCENT <= 100:
    raise ValueError('PROGRESS_EVERY_PERCENT debe estar entre 1 y 100.')

splits = validate_sem1_splits(SPLIT_PATHS, expected_sizes=EXPECTED_SPLIT_SIZES)
SPLIT_FINGERPRINTS = split_fingerprints(splits)
print({name: len(frame) for name, frame in splits.items()})
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu, 'bf16': BF16})
print({'device': available_device(), 'seed': SEED, 'progress_every_percent': PROGRESS_EVERY_PERCENT})
if gpu is None:
    print('AVISO: no se detectó GPU CUDA; el entrenamiento correrá en CPU y será muy lento.')

## Crear o retomar las corridas

Cada variante tiene una carpeta de corrida con el mejor checkpoint de cada modelo, su historial de pérdidas, las predicciones y un manifiesto local. Antes de entrenar se revisan todas las carpetas: si alguna se hizo con otros datos, otra variante u otra semilla, el notebook se detiene aquí y no a mitad del entrenamiento. Con `RESUME = False`, el notebook se niega a mezclar una corrida anterior con una nueva.

In [ ]:
from claim_extraction.workflow import assert_run_matches

manifest_models = {model_id: {
    'display_name': get_model(model_id).display_name,
    'base_checkpoint': get_model(model_id).base_checkpoint,
    'training': get_model(model_id).training,
    'generation': get_model(model_id).generation,
} for model_id in MODELS_TO_RUN}


def prepare_run(variant):
    """Crea o retoma la carpeta de una variante y registra su manifiesto."""
    run_dir = RUN_DIRS[variant]
    manifest_path = run_dir / 'run_manifest.json'
    if run_dir.exists() and any(run_dir.iterdir()) and not RESUME:
        raise FileExistsError(f'La corrida ya existe en {run_dir}. Activa RESUME o cambia de carpeta; no se sobrescribe.')
    for directory in (run_dir / 'models', run_dir / 'predictions'):
        directory.mkdir(parents=True, exist_ok=True)
    if manifest_path.is_file():
        run_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        # RESUME reutiliza checkpoints: deben venir de estos mismos datos, variante y semilla.
        assert_run_matches(run_manifest, dataset_name=DATASET_NAME, variant=variant, fingerprints=SPLIT_FINGERPRINTS, seed=SEED)
        # Permite entrenar los modelos por partes: cada ejecución agrega los suyos.
        run_manifest['models'] = {**run_manifest.get('models', {}), **manifest_models}
    else:
        run_manifest = {
            'dataset_name': DATASET_NAME,
            'run_name': run_dir.name,
            'started_at_utc': utc_now(),
            'device': available_device(),
            'platform': platform.platform(),
            'seed': SEED,
            'variant': variant,
            'variant_config': EXPERIMENT_VARIANTS[variant],
            'early_stopping_patience': EARLY_STOPPING_PATIENCE,
            'precision': 'bf16' if BF16 else 'fp32',
            'batch_size': BATCH_SIZE,
            'split_rows': {name: len(frame) for name, frame in splits.items()},
            'split_fingerprints': SPLIT_FINGERPRINTS,
            'models': manifest_models,
        }
    write_json(manifest_path, run_manifest)
    return run_dir


for variant in VARIANTS_TO_RUN:
    print(f'Corrida preparada: {prepare_run(variant).relative_to(ROOT)}')

## Ejecutar entrenamiento e inferencia

Esta celda puede tardar muchas horas: recorre las variantes y, dentro de cada una, los modelos. Cada modelo se ajusta con `train.csv`, selecciona su checkpoint por pérdida de validación y luego genera un CSV de predicciones sobre los casos de prueba. Al terminar cada modelo se guarda `best/` con `trainer_state.json`, se borran los `checkpoint-*` intermedios y se libera la memoria antes del siguiente.

Si un modelo falla, por ejemplo por falta de memoria, el error queda en el `run_status.json` de su variante y el notebook sigue con los demás. La última celda lista los fallos. Para reintentar solo lo que falta, vuelve a ejecutar el notebook con `RESUME = True`: lo ya terminado se reutiliza.

In [ ]:
# Todos los modelos se entrenan con el mismo train/validation y predicen el mismo test fijo.
failures = []
for variant in VARIANTS_TO_RUN:
    variant_config = EXPERIMENT_VARIANTS[variant]
    run_dir = RUN_DIRS[variant]
    status_path = run_dir / 'run_status.json'
    status = {'status': 'running', 'variant': variant, 'updated_at_utc': utc_now(), 'models': {}}
    write_json(status_path, status)
    print(f'=== Variante {variant} ===', flush=True)
    for model_id in MODELS_TO_RUN:
        model_dir = run_dir / 'models' / model_id
        checkpoint = model_dir / 'best'
        prediction_path = run_dir / 'predictions' / f'{model_id}.csv'
        try:
            # best/ solo cuenta como terminado si también se guardó su historial de pérdidas.
            if RESUME and checkpoint.is_dir() and (model_dir / 'trainer_state.json').is_file():
                print(f'{variant}/{model_id}: checkpoint existente; se reutiliza.')
            else:
                print(f'{variant}/{model_id}: entrenamiento iniciado.', flush=True)
                status['models'][model_id] = {'status': 'training'}
                write_json(status_path, status)
                manifest = train_seq2seq_model(
                    model_id=model_id,
                    train=splits['train'],
                    validation=splits['validation'],
                    output_dir=model_dir,
                    seed=SEED,
                    batch_size=BATCH_SIZE,
                    cache_dir=CACHE_DIR,
                    progress_reporter=lambda percent, current_step, total_steps, current_model=model_id: show_training_progress(current_model, percent, current_step, total_steps),
                    progress_every_percent=PROGRESS_EVERY_PERCENT,
                    gradient_checkpointing=model_id in GRADIENT_CHECKPOINTING_MODELS,
                    epochs=variant_config['epochs'],
                    learning_rate=variant_config.get('learning_rate'),
                    early_stopping_patience=EARLY_STOPPING_PATIENCE,
                    lora=variant_config['lora'],
                    bf16=BF16,
                )
                print(f"{variant}/{model_id}: entrenamiento terminado en {manifest['device']} (mejor época {manifest['best_epoch']} de {manifest['epochs_run']}).")
                # Las predicciones anteriores ya no corresponden al checkpoint nuevo.
                prediction_path.unlink(missing_ok=True)
                release_accelerator_memory()

            if RESUME and prediction_path.is_file():
                print(f'{variant}/{model_id}: predicciones existentes; se reutilizan.')
            else:
                status['models'][model_id] = {'status': 'generating', 'checkpoint': str(checkpoint)}
                write_json(status_path, status)
                generated = generate_predictions(
                    model_id=model_id,
                    checkpoint=checkpoint,
                    candidates=splits['test'],
                    output_path=prediction_path,
                    batch_size=BATCH_SIZE,
                    cache_dir=CACHE_DIR,
                )
                if len(generated) != len(splits['test']):
                    prediction_path.unlink(missing_ok=True)
                    raise AssertionError(f"{model_id} no generó las {len(splits['test'])} predicciones esperadas.")

            status['models'][model_id] = {
                'status': 'ready_for_evaluation',
                'checkpoint': str(checkpoint),
                'predictions': str(prediction_path),
            }
        except Exception as exc:
            # Se guarda solo el texto del error para no retener el modelo en memoria.
            error = f'{type(exc).__name__}: {exc}'
            status['models'][model_id] = {'status': 'failed', 'error': error}
            failures.append(f'{variant}/{model_id}: {error}')
            print(f'{variant}/{model_id}: FALLÓ ({error}). Se sigue con el siguiente modelo.', flush=True)
        finally:
            status['updated_at_utc'] = utc_now()
            write_json(status_path, status)
            release_accelerator_memory()

    failed = any(details['status'] == 'failed' for details in status['models'].values())
    status['status'] = 'failed' if failed else 'ready_for_evaluation'
    status['updated_at_utc'] = utc_now()
    write_json(status_path, status)

print('Entrenamiento recorrido. Revisa la tabla de la siguiente celda.')

In [ ]:
import pandas as pd

status_table = pd.DataFrame([
    {'variante': variant, 'modelo': model_id, **details}
    for variant in VARIANTS_TO_RUN
    for model_id, details in json.loads((RUN_DIRS[variant] / 'run_status.json').read_text(encoding='utf-8'))['models'].items()
])
display(status_table)
if failures:
    raise RuntimeError(
        'Fallaron estos entrenamientos:\n- ' + '\n- '.join(failures)
        + '\nCorrige la causa y vuelve a ejecutar el notebook con RESUME = True: solo se reintenta lo que falta.'
    )
print('Listo. Abre el notebook 04 para calcular las métricas sobre test.csv.')